In [1]:
import pandas as pd

In [2]:
files = [
    "../data_final/top_movies_with_imdb_0_1000.csv",
    "../data_final/top_movies_with_imdb_1000_2000.csv",
    "../data_final/top_movies_with_imdb_2000_3000.csv",
    "../data_final/top_movies_with_imdb_3000_4000.csv",
    "../data_final/top_movies_with_imdb_4000_4480.csv"
]

df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

df.to_csv("../data_final/top_movies_complete.csv", index=False)

print(df.shape)

(4480, 38)


In [3]:
df = pd.read_csv('../data_final/top_movies_complete.csv')
df.head()

,movie_name,release_date,production_budget_usd,domestic_gross_usd,worldwide_gross_usd,domestic_box_office_usd,international_box_office_usd,worldwide_box_office_usd,opening_weekend_usd,legs,...,roi_domestic,log_domestic_box_office,log_worldwide_box_office,log_budget,log_opening_weekend,log_opening_theaters,log_max_theaters,imdb_id,imdb_rating,imdb_votes
0,Star Wars Ep. VII: The Force Awakens,2015-12-16,533200000,936662225,2056046835,936662225.0,1.119385e+09,2.056047e+09,247966675.0,3.78,...,1.756681,20.657833,21.444051,20.094407,19.328805,8.327243,8.327243,NaN,NaN,NaN
1,Avatar: The Way of Water,2022-12-09,460000000,684075767,2315589775,684075767.0,1.631514e+09,2.315590e+09,134100226.0,5.10,...,1.487121,20.343579,21.562930,19.946737,18.714098,8.343554,8.375860,tt1630029,7.5,"601,100"
2,Indiana Jones and the Dial of Destiny,2023-06-28,402300000,174480468,383963057,174480468.0,2.094826e+08,3.839631e+08,60368101.0,2.89,...,0.433707,18.977323,19.766057,19.812709,17.915971,8.434029,8.434029,tt1462764,6.5,"230,035"
3,Avengers: Endgame,2019-04-23,400000000,858373000,2748242781,858373000.0,1.889870e+09,2.748243e+09,357115007.0,2.40,...,2.145932,20.570549,21.734228,19.806975,19.693568,8.447414,8.447414,tt4154796,8.4,"1,442,446"
4,Pirates of the Caribbean: On Stranger Tides,2011-05-20,379000000,241071802,1045713802,241071802.0,8.046420e+08,1.045714e+09,90151958.0,2.67,...,0.636073,19.300605,20.767966,19.753047,18.317007,8.332308,8.334472,tt1298650,6.6,"606,603"


In [4]:
total = len(df)
matched = df["imdb_id"].notna().sum()
ratings = df["imdb_rating"].notna().sum()

print("Total rows:", total)
print("Rows with IMDb ID:", matched)
print("Rows with IMDb rating:", ratings)
print("IMDb ID match rate:", matched / total)
print("IMDb rating coverage:", ratings / total)

Total rows: 4480
Rows with IMDb ID: 3963
Rows with IMDb rating: 3865
IMDb ID match rate: 0.8845982142857143
IMDb rating coverage: 0.8627232142857143


In [5]:
df["imdb_rating_num"] = pd.to_numeric(df["imdb_rating"], errors="coerce")

df["imdb_votes_num"] = (
    df["imdb_votes"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df["imdb_votes_num"] = pd.to_numeric(df["imdb_votes_num"], errors="coerce")

In [6]:
invalid_ratings = df[
    df["imdb_rating_num"].notna() &
    ((df["imdb_rating_num"] < 0) | (df["imdb_rating_num"] > 10))
]

print("Invalid ratings:", len(invalid_ratings))

invalid_ratings[[
    "movie_name",
    "release_year",
    "imdb_id",
    "imdb_rating"
]]

Invalid ratings: 0


,movie_name,release_year,imdb_id,imdb_rating


In [7]:
dupes = df[df["imdb_id"].notna() & df["imdb_id"].duplicated(keep=False)]

print("Duplicate IMDb IDs:", len(dupes))

dupes[[

    "movie_name",

    "release_year",

    "imdb_id",

    "imdb_rating",

    "imdb_votes"

]].sort_values("imdb_id").head(50)

Duplicate IMDb IDs: 4


,movie_name,release_year,imdb_id,imdb_rating,imdb_votes
1218,Super 8,2011.0,tt1650062,7.0,"379,157"
4141,Super,2011.0,tt1650062,7.0,"379,157"
3644,3,2011.0,tt1778304,5.8,"103,694"
3824,Paranormal Activity 3,2011.0,tt1778304,5.8,"103,694"


Dropped all duplicates and the ones where the imdb_id was missing, meaning the pull failed for those variables 

In [8]:
df_clean = df.drop_duplicates(
    subset=["imdb_id"],
    keep=False
)

print("Before:", len(df))
print("After:", len(df_clean))
print("Dropped:", len(df) - len(df_clean))

Before: 4480
After: 3959
Dropped: 521


In [9]:
df_clean['imdb_id'].isna().sum()

np.int64(0)

In [10]:
df_clean[df_clean["imdb_id"].notna()][[
    "movie_name",
    "release_year",
    "imdb_id",
    "imdb_rating",
    "imdb_votes"
]].sample(25, random_state=42)

,movie_name,release_year,imdb_id,imdb_rating,imdb_votes
154,Wild Wild West,1999.0,tt0120891,4.9,"171,971"
1074,Rock Dog,2016.0,tt2822672,6.0,"6,393"
2858,Joe Dirt,2001.0,tt0245686,6.0,"64,259"
755,Shark Tale,2004.0,tt0307453,6.0,"211,362"
339,Fast Five,2011.0,tt1596343,7.3,"430,221"
899,Entrapment,1999.0,tt0137494,6.3,"129,352"
4072,Let There Be Light,2017.0,tt5804314,4.6,"3,374"
427,The Martian,2015.0,tt3659388,8.0,"999,128"
3016,Idle Hands,1999.0,tt0138510,6.3,"49,023"
3504,The Purge: Anarchy,2014.0,tt2975578,6.4,"171,756"


compare with IMDB official ratings

In [11]:
ratings = pd.read_csv(
    "https://datasets.imdbws.com/title.ratings.tsv.gz",
    sep="\t"
)

check = df.merge(
    ratings,
    left_on="imdb_id",
    right_on="tconst",
    how="left"
)

check["rating_diff"] = (
    check["imdb_rating_num"] - check["averageRating"]
).abs()

print(check["rating_diff"].describe())

large_diff = check[
    check["rating_diff"].notna() &
    (check["rating_diff"] > 0.1)
]

print("Rows with rating difference > 0.1:", len(large_diff))

large_diff[[
    "movie_name",
    "release_year",
    "imdb_id",
    "imdb_rating_num",
    "averageRating",
    "rating_diff",
    "imdb_votes_num",
    "numVotes"
]].head(50)

count    3864.000000
mean        0.012138
std         0.067653
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.900000
Name: rating_diff, dtype: float64
Rows with rating difference > 0.1: 163


,movie_name,release_year,imdb_id,imdb_rating_num,averageRating,rating_diff,imdb_votes_num,numVotes
262,Wicked,2024.0,tt1262426,7.4,7.3,0.1,191746.0,242490.0
306,Penguins of Madagascar,2014.0,tt1911658,6.6,6.7,0.1,113406.0,116106.0
314,Mary Poppins Returns,2018.0,tt5028340,6.7,6.6,0.1,98768.0,101071.0
337,The Da Vinci Code,2006.0,tt0382625,6.6,6.7,0.1,480857.0,486598.0
432,"10,000 B.C.",2008.0,tt0443649,5.1,5.2,0.1,139178.0,139581.0
485,Land of the Lost,2009.0,tt0457400,5.3,5.4,0.1,78326.0,80079.0
488,Catwoman,2004.0,tt0327554,3.4,3.5,0.1,131602.0,132416.0
551,Puss in Boots: The Last Wish,2022.0,tt3915174,7.8,7.9,0.1,218530.0,225931.0
557,Charlie's Angels,2000.0,tt0160127,5.6,5.7,0.1,206779.0,206803.0
564,DC League of Super Pets,2022.0,tt8912936,6.9,6.8,0.1,80000.0,80141.0


In [12]:
df_clean.to_csv("../data_final/top_movies_imdb_final.csv", index=False)